# Entity representations

Run notebook 3_01 first.
**Input:** article entity annotations and training-release entity embeddings.
**Output:** article_entity100, annotation links, missing-vector flags and coverage reports.
Deduplicate entity IDs across title/abstract; mean-pool matched unit vectors and normalise.
The dev release is audited for compatibility, never merged into the embedding source.

Run the cells in order. Code must be available in this runtime, including `src/newsrec`.
For Colab, upload the full checkout or mount its folder on Drive; opening only the notebook does not upload the package.
Set `NEWSREC_CODE` before the setup cell if the checkout is elsewhere. Set `DATA_ROOT` for separate datasets.
Inputs are prepared parquet files under `1_Data_Preparation/processed`; outputs go to `3_Representations/artifacts`.
The calculation cells below are passed to shared storage helpers.

In [1]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Code: run from the Group 6 checkout, or set NEWSREC_CODE to that folder.
CODE_ROOT = (
    Path(os.environ["NEWSREC_CODE"]).expanduser().resolve()
    if os.environ.get("NEWSREC_CODE")
    else next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/newsrec").is_dir()), None)
)
if CODE_ROOT is None or not (CODE_ROOT / "src/newsrec").is_dir():
    raise FileNotFoundError("Open the project checkout, or set NEWSREC_CODE to the folder containing src/newsrec/.")
sys.path.insert(0, str(CODE_ROOT / 'src'))

# Data: use the checkout by default. For Drive, mount it first and set DATA_ROOT.
DATA_ROOT = None  # Colab example: '/content/drive/MyDrive/BT4222'
ROOT = Path(DATA_ROOT or os.getenv('NEWSREC_ROOT') or CODE_ROOT / 'runs/mindsmall-epoch').expanduser().resolve()
ARTIFACTS = ROOT / '3_Representations/artifacts'
print('Code:', CODE_ROOT)
print('Data and outputs:', ROOT)

from newsrec.common import DEFAULT_SETTINGS, Settings, normalise, saved_stage_ready
from newsrec.representations import build_entities, parse_entities

Code: /Users/bryan03/Desktop/BT4222-Group-6-
Data and outputs: /Users/bryan03/Desktop/BT4222-Group-6-


In [2]:
SETTINGS = Settings()  # Fixed seed and dimensions; use the same settings in all stages.
REBUILD = False  # Set True after editing a calculation cell. Shared code changes invalidate caches.
# Terminal override applies only to this execution; the saved preference stays False.
REBUILD = REBUILD or os.environ.get("NEWSREC_FORCE_REBUILD") == "1"

## 1. Parse and pool

Parse JSON; deduplicate Wikidata IDs across title and abstract; normalise each matched entity vector,
mean-pool, then normalise the article vector. Empty/unmatched articles use zero plus a validity mask.
The dev lookup is audited for coordinate differences, never merged automatically into the train space.
Zero vectors, malformed rows, nonfinite coordinates and duplicate entity IDs are checked explicitly.

### Article entity pooling

The storage helper reads and normalises the **training-release** entity vectors only. This function deduplicates annotations, pools matched vectors, and records missing coverage. Dev vectors are audited separately and never merged.

In [3]:
def pool_article_entities(index, lookup, vectors, cfg=DEFAULT_SETTINGS):
    article = np.zeros((len(index) + 1, cfg.entity_dim), np.float32)
    links, quality, article_sets = ([], [], [])
    # Process each article independently; preserve the shared article-row mapping.
    for row in index.itertuples(index=False):
        title_ids, ts = parse_entities(row.title_entities) # Parse title entities from the Mind dataset
        abstract_ids, aps = parse_entities(row.abstract_entities) # Parse abstract entities
        all_ids = sorted(set(title_ids) | set(abstract_ids))
        article_sets.append(set(all_ids))
        matched = [
            lookup[x]
            for x in all_ids
            if x in lookup and np.linalg.norm(vectors[lookup[x]]) > 0
        ]
        if matched:
            article[row.article_row] = normalise(vectors[matched].mean(axis=0))
        quality.append(
            {
                "news_code": row.news_code,
                "title_entity_status": ts,
                "abstract_entity_status": aps,
                "n_title_entities": len(title_ids),
                "n_abstract_entities": len(abstract_ids),
                "n_annotated_entities": len(all_ids),
                "n_valid_entities": len(matched),
                "entity_coverage": len(matched) / max(1, len(all_ids)),
                "has_entity_vector": bool(np.linalg.norm(article[row.article_row]) > 0),
            }
        )
        for key in all_ids:
            links.append(
                {
                    "news_code": row.news_code,
                    "entity_id": key,
                    "entity_row": lookup.get(key, 0),
                    "in_title": key in title_ids,
                    "in_abstract": key in abstract_ids,
                }
            )
    return article, links, quality, article_sets

In [4]:
if REBUILD or not saved_stage_ready(ROOT, SETTINGS, "entity"):
    coverage = build_entities(ROOT, SETTINGS, article_pooler=pool_article_entities)
else:
    print("Loaded saved entity representations.")
    report = pd.read_csv(ROOT / "results/representation_coverage.csv")
    coverage = report[report.stage.eq("entity")]
display(coverage)

,split,representation,cohort,denominator,valid,fallback,valid_fraction
0,train,entity_ids,split_catalogue,42628,41231,1397,0.967228
1,train,article_entity,split_catalogue,98671,86704,11967,0.878718
2,validation,entity_ids,split_catalogue,34512,33391,1121,0.967519
3,validation,article_entity,split_catalogue,71991,62913,9078,0.873901
4,test,entity_ids,split_catalogue,34391,32689,1702,0.950510
5,test,article_entity,split_catalogue,72023,62769,9254,0.871513


## 2. Audit coverage at the right denominator

Entity-ID lookup coverage, article-vector coverage and user-profile coverage are different quantities.
The preparation report's test lookup coverage uses the dev file; this notebook reports the canonical
train lookup for every split. It separately records whether the dev coordinates match.

In [5]:
quality = pd.read_parquet(ARTIFACTS / "entity_quality.parquet")
display(quality.head())
display(
    quality.groupby(["title_entity_status", "abstract_entity_status"])
    .size()
    .rename("articles")
    .reset_index()
)
meta = json.loads((ARTIFACTS / "entity_manifest.json").read_text())
display(meta["dev_compatibility"])
vectors = np.load(ARTIFACTS / "article_entity100.npy", mmap_mode="r")
assert vectors.shape == (len(quality) + 1, SETTINGS.entity_dim)
assert np.isfinite(vectors).all() and np.all(vectors[0] == 0)
assert np.array_equal(
    np.linalg.norm(vectors[1:], axis=1) > 0, quality.has_entity_vector.to_numpy()
)
print("Article entity matrix MB:", vectors.nbytes / 1e6)

# Output: article-level fallback counts; never compare entity-ID coverage with this denominator.
norms = np.linalg.norm(vectors, axis=1)
assert vectors.dtype == np.float32
assert np.allclose(norms[norms > 0], 1, atol=1e-4)
index = pd.read_parquet(ARTIFACTS / 'article_index.parquet')
text = np.load(ARTIFACTS / 'text64.npy', mmap_mode='r')
has_text = np.linalg.norm(text[index.article_row], axis=1) > 0
has_entity = norms[index.article_row] > 0
fallback = pd.DataFrame({
    'news_code': index.news_code,
    'has_text': has_text,
    'has_entity': has_entity,
    'fallback': np.select([has_text & has_entity, has_text, has_entity],
                          ['text_and_entity', 'text_only', 'entity_only'], default='neither'),
})
fallback.to_parquet(ARTIFACTS / 'article_fallback.parquet', index=False)
display(fallback.groupby('fallback').size().rename('articles').reset_index())

,news_code,title_entity_status,abstract_entity_status,n_title_entities,n_abstract_entities,n_annotated_entities,n_valid_entities,entity_coverage,has_entity_vector
0,1,present,present,2,3,3,3,1.0,True
1,2,present,present,1,3,4,4,1.0,True
2,4,present,present,1,6,7,7,1.0,True
3,5,empty,present,0,1,1,1,1.0,True
4,6,present,present,2,3,3,3,1.0,True


,title_entity_status,abstract_entity_status,articles
0,empty,empty,12187
1,empty,present,14544
2,present,empty,13933
3,present,present,63487


{'shared_ids': 32689,
 'max_coordinate_difference': 0.0,
 'dev_only_ids': 621,
 'policy': 'audit only; no dev-only vectors merged'}

Article entity matrix MB: 41.6608


,fallback,articles
0,text_and_entity,91452
1,text_only,12699


## Before -> after: one real article

The cell below picks a real article from the saved artifacts and shows its raw dataset fields
(BEFORE) next to what this notebook produced for it (AFTER): the pooled 100-dim vector row,
its quality record, and its annotation links. No model is trained here - it is lookup + mean-pooling
over the supplied training-release entity vectors.

In [6]:
index = pd.read_parquet(ARTIFACTS / "article_index.parquet")
quality = pd.read_parquet(ARTIFACTS / "entity_quality.parquet")
links = pd.read_parquet(ARTIFACTS / "article_entity_links.parquet")
vectors = np.load(ARTIFACTS / "article_entity100.npy", mmap_mode="r")

# Pick a readable real example: first valid article with 1-3 annotated entities.
example = (
    quality[quality.has_entity_vector]
    .query("1 <= n_annotated_entities <= 3")
    .iloc[0]
)
row = index[index.news_code.eq(example.news_code)].iloc[0]
vec = np.asarray(vectors[row.article_row], dtype=np.float32)

print("BEFORE - raw dataset fields, entities are stored as raw text JSON strings")
print(f"  news_code: {row.news_code}")
print(f"  title: {row.title}")
print(f"  abstract: {row.abstract}")
print(f"  title_entities: {row.title_entities}")
print(f"  abstract_entities: {row.abstract_entities}")

print("\nAFTER - what this notebook produced")
print(
    f"  article_entity100[row {row.article_row}] = 100-dim vector, "
    f"norm={np.linalg.norm(vec):.3f}, first 5 dims={vec[:5]}"
)
print(
    f"  quality: n_annotated={example.n_annotated_entities}, "
    f"n_valid={example.n_valid_entities}, "
    f"coverage={example.entity_coverage:.2f}, "
    f"has_entity_vector={example.has_entity_vector}"
)
print("  annotation links (entity_row=0 means not in the train lookup):")
display(links[links.news_code.eq(example.news_code)])

BEFORE - raw dataset fields, entities are stored as raw text JSON strings
  news_code: 1
  title: Texans defensive tackle D.J. Reader is taking advantage of his opportunities
  abstract: Houston Texans defensive tackle D.J. Reader is taking advantage of opportunities given by defensive end J.J. Watt.
  title_entities: [{"Label": "D. J. Reader", "Type": "P", "WikidataId": "Q24007178", "Confidence": 1.0, "OccurrenceOffsets": [24], "SurfaceForms": ["D.J. Reader"]}, {"Label": "Houston Texans", "Type": "O", "WikidataId": "Q223514", "Confidence": 1.0, "OccurrenceOffsets": [0], "SurfaceForms": ["Texans"]}]
  abstract_entities: [{"Label": "D. J. Reader", "Type": "P", "WikidataId": "Q24007178", "Confidence": 1.0, "OccurrenceOffsets": [32], "SurfaceForms": ["D.J. Reader"]}, {"Label": "Houston Texans", "Type": "O", "WikidataId": "Q223514", "Confidence": 1.0, "OccurrenceOffsets": [0], "SurfaceForms": ["Houston Texans"]}, {"Label": "J. J. Watt", "Type": "P", "WikidataId": "Q1097511", "Confidence": 

,news_code,entity_id,entity_row,in_title,in_abstract
0,1,Q1097511,5824,False,True
1,1,Q223514,4233,True,True
2,1,Q24007178,36735,True,True


## Usage-weighted coverage

Read each log in batches. Count vectors for displayed entries, clicked entries and each user's supplied history once per split.
`neither` requires a popularity/category fallback downstream; a zero vector must not be treated as a meaningful similarity.
These diagnostics inspect labels only to count click coverage; they do not fit embeddings.

In [7]:
from newsrec.audits import usage_coverage
from newsrec.common import measure

# Output: usage_coverage.csv. This is a full log scan; its cost is recorded separately.
with measure(ROOT, 'representation_coverage_audit'):
    display(usage_coverage(ROOT, SETTINGS))
display(pd.read_csv(ROOT / 'results/representation_cost.csv').tail())

,split,cohort,entries,both,text_only,entity_only,neither,text_coverage,entity_coverage
0,train,slate,70314387,54801466,15512921,0,0,1.0,0.779378
1,train,click,2868998,2365044,503954,0,0,1.0,0.824345
2,train,history,12874380,10513177,2361203,0,0,1.0,0.816597
3,validation,slate,13192987,10496534,2696453,0,0,1.0,0.795615
4,validation,click,514658,422109,92549,0,0,1.0,0.820174
5,validation,history,6244894,5101099,1143795,0,0,1.0,0.816843
6,test,slate,14085557,11115099,2970458,0,0,1.0,0.789113
7,test,click,574845,495300,79545,0,0,1.0,0.861624
8,test,history,6560336,5366618,1193718,0,0,1.0,0.818040


,stage,wall_seconds,sampled_peak_process_rss_mb,platform,cpu,logical_cpus,ram_gb
17,ranking_features,13.726263,1956.347904,macOS-26.6.2-arm64-arm-64bit,arm,10,17.179869
18,ranking_features,13.148860,1828.306944,macOS-26.6.2-arm64-arm-64bit,arm,10,17.179869
19,text,6.703648,1223.868416,macOS-26.6.2-arm64-arm-64bit,arm,10,17.179869
20,entity,3.110573,1042.087936,macOS-26.6.2-arm64-arm-64bit,arm,10,17.179869
21,representation_coverage_audit,7.528604,1749.123072,macOS-26.6.2-arm64-arm-64bit,arm,10,17.179869


## Handoff

`article_entity100.npy` shares the article-row map with the text vectors. Entity and text coordinates
remain separate. `article_entity_links.parquet` supplies annotation links (including unmatched IDs)
for later graph construction; these annotation links are not evidence that an embedding exists.
The following notebook produces supplied-only entity user profiles for the clustering owner.

**Known limitation - pooled vectors can collide.** Mean-pooling compresses each article's entity
*set* into one centroid, so (1) any two articles annotating the same entity set get **identical**
vectors, and (2) different entity sets whose entities cluster in the same region of embedding space
(e.g. two different tennis players) get **similar** vectors - the mean washes out which specific
entities were mentioned. This is accepted because the entity vector is a coarse topic feature, not
the whole representation: `title64`/`text64` and `title_token_ids` keep word-level distinction
between articles, and `article_entity_links.parquet` preserves the exact per-article entity set for
consumers that need it (e.g. multi-hot entity features or graph features).